# Classification de texte avec des RNN

Dans ces travaux pratiques, nous allons retrouver l'auteur d'une phrase avec des réseaux récurrents, écrits en PyTorch et entraînés avec Lightning. Le corpus est celui des slides : les phrases de 2 œuvres de chacun de 9 auteurs anglophones, de 1821 à 1913 (Brontë, Melville, Wilde…).

1. Du texte aux séquences d'indices.
2. Un classifieur récurrent qui ignore le *padding*.
3. L'entraînement d'un GRU bidirectionnel.
4. Sans empaquetage, le GRU lit le *padding*.
5. GRU simple ou bidirectionnel ?
6. Des embeddings GloVe pré-entraînés et figés.
7. Un modèle plus profond.
8. Les auteurs que le modèle confond.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Chaque entraînement y prend quelques minutes, contre un quart d'heure à plus d'une heure sur CPU.

## Installation et téléchargements

PyTorch Lightning, le corpus, le modèle de spaCy pour l'anglais et les vecteurs GloVe de la partie 6 (un fichier de 822 Mo) : comptez une à deux minutes.

In [ ]:
!pip install -q lightning
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Import des bibliothèques

In [ ]:
import collections
import pathlib
from collections.abc import Iterable, Iterator

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import torch
import tqdm.auto
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, TensorDataset, random_split

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Prétraitements

La cellule suivante prépare les données :

- chaque phrase, découpée par spaCy, est un exemple, et son étiquette est son auteur, de 0 à 8 ;
- toute entité nommée (un mot ou un groupe de mots) est remplacée par son type, première lettre triplée pour en faire un mot nouveau (`PPPERSON`, `DDDATE`…) : le modèle doit reconnaître un style, pas les noms des personnages ;
- les dossiers du corpus portent des noms à coquilles (`Melwille`, `bronteCharlote`) : `AUTEURS` donne le nom de chaque auteur ;
- une phrase sans aucun mot (un crochet « [ » isolé par spaCy, de la ponctuation seule) est écartée : vides une fois découpées en mots, ces 1 763 phrases auraient une longueur nulle, que refuse `pack_padded_sequence` (partie 2) ;
- 70 % des phrases servent à l'entraînement (`X_train_raw`, `y_train`), 30 % au test (`X_test_raw`, `y_test`), toujours les mêmes.

Elle prend environ deux minutes.

In [ ]:
# Dossier du corpus → nom de l'auteur
AUTEURS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def remplace_entites(jetons: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for jeton in jetons:
    if jeton.ent_iob_ == "O":
      yield jeton.text
    elif jeton.ent_iob_ == "B":
      prefixe = jeton.ent_type_[0]
      yield f"{prefixe}{prefixe}{jeton.ent_type_}"


def lit_donnees(dossier: pathlib.Path) -> tuple[list[str], list[str]]:
  textes = []
  auteurs = []
  # <auteur>/<œuvre>/<chapitre>.txt
  chemins = sorted(dossier.glob("*/*/*.txt"))
  contenus = (p.read_text(encoding="utf8").replace("\n", " ") for p in chemins)
  for chemin, doc in tqdm.auto.tqdm(
    zip(chemins, nlp.pipe(contenus)), total=len(chemins)
  ):
    auteur = AUTEURS[chemin.parent.parent.name]
    for phrase in doc.sents:
      texte = " ".join(remplace_entites(phrase))
      # Une phrase sans lettre ni chiffre serait vide une fois découpée en mots
      if any(caractere.isalnum() for caractere in texte):
        auteurs.append(auteur)
        textes.append(texte)
  return textes, auteurs


textes, auteurs = lit_donnees(pathlib.Path("classical-authors"))
encodeur_etiquettes = sklearn.preprocessing.LabelEncoder()
y = torch.tensor(encodeur_etiquettes.fit_transform(auteurs))
print(
  f"{len(textes)} phrases ; étiquettes 0 à 8 : {', '.join(encodeur_etiquettes.classes_)}"
)

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  textes, y, test_size=0.3, random_state=0
)
print(f"Entraînement : {len(X_train_raw)} phrases, test : {len(X_test_raw)}")
for texte, etiquette in zip(X_train_raw[:3], y_train[:3]):
  print(f"{encodeur_etiquettes.classes_[etiquette]} : {texte}")

## 1. Du texte aux séquences d'indices

La classe `Tokenizer` de la cellule suivante est celle des slides. Elle numérote les mots d'un corpus :

- un mot est une suite de caractères entre deux espaces, une fois le texte mis en minuscules et la ponctuation retirée ;
- `apprend_vocabulaire(textes)` construit le vocabulaire `vocabulaire`, la liste des mots dans l'ordre de leurs indices : `<pad>` (indice 0, le *padding*), `<unk>` (indice 1, tout mot hors du vocabulaire), puis les `max_jetons - 2` mots les plus fréquents de `textes`, du plus au moins fréquent ;
- `convertit_en_sequences(textes)` remplace chaque mot de chaque texte par son indice.

In [ ]:
class Tokenizer:
  """Associe aux mots les plus fréquents d'un corpus des indices, à partir de 2."""

  # Caractères remplacés par des espaces avant le découpage en mots
  filtres = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

  def __init__(self, max_jetons: int) -> None:
    self.max_jetons = max_jetons
    self.table = str.maketrans(self.filtres, " " * len(self.filtres))
    self.vocabulaire: list[str] = []
    self.indice_mots: dict[str, int] = {}

  def decoupe(self, texte: str) -> list[str]:
    return texte.lower().translate(self.table).split()

  def apprend_vocabulaire(self, textes: Iterable[str]) -> None:
    comptes = collections.Counter(
      mot for texte in textes for mot in self.decoupe(texte)
    )
    # Indices 0 et 1 réservés au padding et aux mots inconnus
    plus_frequents = [mot for mot, _ in comptes.most_common(self.max_jetons - 2)]
    self.vocabulaire = ["<pad>", "<unk>", *plus_frequents]
    self.indice_mots = {mot: index for index, mot in enumerate(self.vocabulaire)}

  def convertit_en_sequences(self, textes: Iterable[str]) -> list[list[int]]:
    # Un mot hors du vocabulaire devient 1, <unk>
    return [
      [self.indice_mots.get(mot, 1) for mot in self.decoupe(texte)] for texte in textes
    ]

La cellule suivante transforme les phrases en tenseurs d'indices de mots, comme dans les slides : un `Tokenizer` de 20 000 mots, appris sur les seules phrases d'entraînement, puis `vectorise`, qui tronque la séquence d'indices de chaque phrase aux 150 premiers et complète les plus courtes par des 0 à la fin (le *padding*), jusqu'à la longueur de la plus longue.

In [ ]:
tokenizer = Tokenizer(max_jetons=20_000)  # les 20 000 mots les plus fréquents
tokenizer.apprend_vocabulaire(X_train_raw)  # vocabulaire de l'entraînement seul
vocabulaire = tokenizer.vocabulaire


def vectorise(textes: list[str]) -> torch.Tensor:
  # 150 indices au plus par phrase
  sequences = [
    torch.tensor(sequence[:150])
    for sequence in tokenizer.convertit_en_sequences(textes)
  ]
  # Des 0 à la fin, jusqu'à la longueur de la plus longue séquence
  return pad_sequence(sequences, batch_first=True, padding_value=0)


X_train = vectorise(X_train_raw)
X_test = vectorise(X_test_raw)

print(tuple(X_train.shape))
print(X_train_raw[0])
print(X_train[0, :20])
print(" | ".join(vocabulaire[:10]))

### Combien de mots garder ?

Les deux choix de la cellule précédente, 150 indices par phrase et un vocabulaire de 20 000 mots, se décident d'après les données. Mesurez-les :

- `longueurs`, le nombre de mots de chaque phrase d'entraînement : les indices non nuls de sa ligne de `X_train` (une phrase de plus de 150 mots y compte pour 150) ;
- l'histogramme de `longueurs`, sa médiane et son 99e centile ;
- la part des phrases d'entraînement tronquées, celles qui remplissent leurs 150 indices ;
- la part des mots des phrases de test hors du vocabulaire (l'indice 1), et celle des phrases de test qui en ont au moins un.

Quelle longueur maximale choisiriez-vous d'après cette distribution ? Que perd-on avec 150 indices, et que coûtent-ils ? Les 20 000 mots du vocabulaire suffisent-ils ?

**Documentation utile** :

- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `(X_train != 0).sum(dim=1).numpy()` : le nombre d'indices non nuls de chaque ligne, en tableau NumPy
- `plt.hist(longueurs, bins=75)` : 75 barres de 2 mots de large ; `plt.yscale("log")` rend visibles les phrases longues, rares
- `np.median(longueurs)`, `np.percentile(longueurs, 99)` : la moitié des phrases sont plus courtes que la médiane, 99 sur 100 que le 99e centile
- `(longueurs == 150).mean()` : la part des phrases tronquées
- `(X_test == 1).sum() / (X_test != 0).sum()` : la part des mots inconnus ; `(X_test == 1).any(dim=1)` dit pour chaque phrase si elle en a au moins un

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
longueurs = (X_train != ...).sum(dim=1).numpy()
plt.hist(longueurs, bins=...)
plt.yscale("log")
plt.xlabel("Mots par phrase")
plt.ylabel("Phrases")
plt.show()
print(f"Médiane : {np.median(...):.0f} mots ; 99e centile : {np.percentile(...):.0f}")
print(f"Phrases tronquées : {(longueurs == ...).mean():.1%}")
print(f"Mots inconnus en test : {(X_test == ...).sum() / (X_test != ...).sum():.1%}")
print(f"Phrases avec un mot inconnu : {(X_test == ...).any(dim=1).float().mean():.0%}")
```

#### Solution

In [ ]:
longueurs = (X_train != 0).sum(dim=1).numpy()  # mots de chaque phrase, 150 au plus
plt.hist(longueurs, bins=75)
plt.yscale("log")  # les phrases longues sont rares
plt.xlabel("Mots par phrase")
plt.ylabel("Phrases")
plt.show()
print(
  f"Médiane : {np.median(longueurs):.0f} mots ; "
  f"99e centile : {np.percentile(longueurs, 99):.0f}"
)
print(f"Phrases tronquées : {(longueurs == 150).mean():.1%}")
print(f"Mots inconnus en test : {(X_test == 1).sum() / (X_test != 0).sum():.1%}")
print(f"Phrases avec un mot inconnu : {(X_test == 1).any(dim=1).float().mean():.0%}")

# Une phrase compte 17 mots en médiane, et 99 % en ont moins de 100 : le 99e centile,
# 100 indices, suffirait. 150 n'en tronquent que 0,2 %, et les 50 indices de plus ne
# coûtent ni justesse ni presque de calcul : le GRU saute le padding (partie 2). Les 20 000 mots gardés, sur les 31 785 des phrases
# d'entraînement, couvrent 98 % des mots de test : 1,8 % sont inconnus (des mots rares,
# des noms que spaCy n'a pas reconnus). Mais 27 % des phrases de test en ont au moins
# un : un vocabulaire plus grand en sauverait quelques-uns, au prix d'une table
# d'embeddings plus grande, pour des mots trop rares pour y être bien appris.

## 2. Un classifieur récurrent qui ignore le *padding*

Le module Lightning `ClassifieurTexte` de la cellule suivante est celui des slides : il enveloppe un modèle PyTorch, `modele`, qui donne un score (un logit) à chacun des 9 auteurs, l'entraîne avec la perte d'entropie croisée et l'optimiseur Adam, et mesure sa justesse (*accuracy*) en entraînement, en validation et en test, dans sa méthode `etape`.

In [ ]:
class ClassifieurTexte(lightning.LightningModule):
  """Enveloppe Lightning : prédit l'auteur d'une séquence de mots avec padding."""

  def __init__(self, modele: nn.Module, taux_apprentissage: float = 1e-3) -> None:
    super().__init__()
    self.modele = modele
    self.taux_apprentissage = taux_apprentissage
    # Une phrase d'exemple, pour les formes des tenseurs du résumé du modèle :
    # 20 mots inconnus (des 1), car une ligne de 0 serait vide
    self.example_input_array = torch.ones(1, 20, dtype=torch.long)

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    return self.modele(sequences)

  def etape(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    sequences, auteurs = batch
    logits = self(sequences)
    perte = nn.functional.cross_entropy(logits, auteurs)
    justesse = (logits.argmax(dim=1) == auteurs).float().mean()
    # Moyennes de chaque epoch, pondérées par la taille des batchs
    self.log_dict(
      {f"{stage}_loss": perte, f"{stage}_accuracy": justesse},
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.etape(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> None:
    self.etape(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> None:
    self.etape(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.taux_apprentissage)

### Le modèle récurrent

La classe `GRUTexte` de la cellule suivante est le classifieur des slides, avec des paramètres de plus pour les modèles des parties suivantes :

- `embedding` : la couche d'embedding, une nouvelle couche `nn.Embedding` de 64 dimensions si elle n'est pas donnée (les vecteurs GloVe de la partie 6) ;
- `nb_unites`, `nb_couches` et `bidirectionnel` : le nombre d'unités, de couches et de sens de lecture du GRU (un GRU simple en partie 5, deux couches en partie 7) ;
- `dropout` : le taux de *dropout* sur les embeddings, et entre les couches du GRU quand il en a plusieurs.

Le GRU ne lit pas le *padding* (slide « Ignorer le *padding* dans le RNN ») : une phrase compte 17 mots en médiane, suivis de plus de 130 zéros dans `X_train`. `forward` lui passe les séquences empaquetées par `pack_padded_sequence`, avec leurs longueurs réelles, le nombre d'indices non nuls de chaque ligne : ses derniers états cachés sont ceux des derniers vrais mots, dans les deux sens de lecture.

In [ ]:
class GRUTexte(nn.Module):
  """Vecteurs des mots, résumé par un GRU, puis un score par auteur."""

  def __init__(
    self,
    embedding: nn.Embedding | None = None,
    nb_unites: int = 64,
    nb_couches: int = 1,
    bidirectionnel: bool = True,
    dropout: float = 0.0,
  ) -> None:
    super().__init__()
    if embedding is None:
      # Un vecteur de 64 nombres par mot, nul pour le padding
      embedding = nn.Embedding(len(vocabulaire), 64, padding_idx=0)
    self.embedding = embedding
    self.dropout = nn.Dropout(dropout)  # sur les embeddings
    self.gru = nn.GRU(
      embedding.embedding_dim,
      nb_unites,
      num_layers=nb_couches,
      bidirectional=bidirectionnel,
      dropout=dropout if nb_couches > 1 else 0.0,  # entre les couches
      batch_first=True,
    )
    directions = 2 if bidirectionnel else 1
    self.tete = nn.Linear(nb_unites * directions, len(encodeur_etiquettes.classes_))

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    vecteurs = self.dropout(self.embedding(sequences))
    # Longueur réelle de chaque séquence : le GRU saute le padding, et ses
    # derniers états cachés sont ceux des derniers vrais mots, dans les deux sens
    longueurs = (sequences != 0).sum(dim=1).cpu()
    empaquetees = pack_padded_sequence(
      vecteurs, longueurs, batch_first=True, enforce_sorted=False
    )
    _, h_n = self.gru(empaquetees)
    if self.gru.bidirectional:
      # Derniers états des sens avant et arrière de la dernière couche
      resume = torch.cat([h_n[-2], h_n[-1]], dim=1)
    else:
      resume = h_n[-1]
    return self.tete(resume)


print(GRUTexte()(X_train[:4]).shape)

## 3. Entraîner un GRU bidirectionnel

La fonction `entraine` de la cellule suivante entraîne un modèle avec Lightning, puis l'évalue :

- le modèle est enveloppé dans un `ClassifieurTexte` ; la norme du gradient est écrêtée à 1 (`gradient_clip_val`), contre l'explosion du gradient, fréquente dans les RNN ;
- des batchs de 256 phrases ; 20 % des phrases d'entraînement en validation, les mêmes à chaque appel ;
- 20 epochs au plus : `EarlyStopping` arrête l'entraînement après 2 epochs sans baisse de la perte de validation, et `ModelCheckpoint` garde les poids de la meilleure, ceux qu'ont le modèle évalué en test et le modèle `modele` après l'appel.

Elle affiche le résumé du modèle et ses courbes de justesse, et renvoie sa justesse de test. La cellule d'après l'applique à un GRU bidirectionnel de 64 unités par sens de lecture.

In [ ]:
def entraine(modele: nn.Module, nom: str, max_epoques: int = 20) -> float:
  classifieur = ClassifieurTexte(modele)
  print(ModelSummary(classifieur, max_depth=-1))
  # Les mêmes 20 % des phrases d'entraînement en validation à chaque appel
  dataset_entrainement, dataset_validation = random_split(
    TensorDataset(X_train, y_train),
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(0),
  )
  # Garde les poids de l'epoch de plus faible perte de validation
  point_controle = ModelCheckpoint(monitor="val_loss")
  trainer = lightning.Trainer(
    max_epochs=max_epoques,
    accelerator="auto",
    devices=1,
    gradient_clip_val=1.0,  # écrêtage de la norme du gradient
    callbacks=[
      # La barre de progression de tqdm, plutôt que celle de rich que Lightning
      # choisit quand rich est installé (Colab) : elle a figé des entraînements
      TQDMProgressBar(),
      EarlyStopping(monitor="val_loss", patience=2),
      point_controle,
    ],
    logger=CSVLogger("logs", name=nom),
    enable_model_summary=False,  # résumé déjà affiché
  )
  trainer.fit(
    classifieur,
    DataLoader(dataset_entrainement, batch_size=256, shuffle=True),
    DataLoader(dataset_validation, batch_size=1024),
  )
  # Test avec les poids de la meilleure epoch, rechargés dans modele
  chargeur_test = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)
  (resultat,) = trainer.test(
    classifieur, chargeur_test, ckpt_path="best", verbose=False
  )
  justesse = resultat["test_accuracy"]

  # Justesses de chaque epoch, écrites par le CSVLogger sur des lignes
  # différentes pour l'entraînement et la validation : on les regroupe
  metriques = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_accuracy", "val_accuracy"])
  )
  meilleur = metriques["val_loss"].idxmin()
  print(
    f"Meilleure epoch : {meilleur + 1} ; justesse d'entraînement "
    f"{metriques['train_accuracy'][meilleur]:.3f}, "
    f"de validation {metriques['val_accuracy'][meilleur]:.3f}, de test {justesse:.3f}"
  )
  plt.plot(metriques.index + 1, metriques["train_accuracy"], label="entraînement")
  plt.plot(metriques.index + 1, metriques["val_accuracy"], label="validation")
  plt.axvline(meilleur + 1, color="gray", linestyle="--", label="meilleure epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Justesse")
  plt.legend()
  plt.show()
  return justesse

In [ ]:
bigru = GRUTexte()
bigru_justesse = entraine(bigru, "bigru")

### Premières conclusions

Au vu des courbes et des justesses, que dire de la qualité de cet apprentissage ? Comparez la justesse de test au hasard (une chance sur 9) et à un modèle qui prédirait toujours l'auteur le plus fréquent, Melville (18 % des phrases).

Votre réponse ici

#### Solution

Le modèle apprend : 55 % de justesse en test, cinq fois le hasard (11 %) et trois fois la classe majoritaire (18 %).

Mais il surapprend : dès la 3e epoch, sa justesse d'entraînement dépasse celle de validation, et l'écart se creuse, jusqu'à 71 % contre 56 % à la meilleure epoch, la 8e. La perte de validation ne baisse presque plus après la 5e epoch, puis remonte : `EarlyStopping` arrête l'entraînement deux epochs plus tard, et `entraine` évalue les poids de la meilleure.

Le modèle a 1,3 million de paramètres, dont 1,28 million dans la table d'embeddings, pour 41 000 phrases d'entraînement : il retient les mots des phrases d'entraînement plutôt que le style de chaque auteur.

## 4. Sans empaquetage, le GRU lit le *padding*

Que gagne-t-on à empaqueter les séquences ? Écrivez `GRUSansEmpaquetage`, une sous-classe de `GRUTexte` dont la méthode `forward` passe au GRU les embeddings des séquences telles quelles, *padding* compris, sans `pack_padded_sequence`. Entraînez avec `entraine` un `GRUSansEmpaquetage()`, bidirectionnel comme le modèle de la partie 3, et comparez leurs justesses de test.

Dans `X_train`, une phrase de 17 mots est suivie de 133 zéros. Par quoi chaque sens de lecture du GRU finit-il sa lecture ? Lequel des deux le *padding* gêne-t-il ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `self.embedding`, `self.dropout`, `self.gru` et `self.tete` : les couches de `GRUTexte`, dont `GRUSansEmpaquetage` hérite ; seule `forward` change
- `_, h_n = self.gru(vecteurs)` : le GRU lit les 150 pas de chaque ligne, *padding* compris
- `torch.cat([h_n[-2], h_n[-1]], dim=1)` : les derniers états des deux sens de lecture, comme dans `GRUTexte`
- `bigru_justesse` contient la justesse de test du GRU bidirectionnel de la partie 3

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class GRUSansEmpaquetage(GRUTexte):
  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    vecteurs = self.dropout(self.embedding(...))
    _, h_n = self.gru(...)
    return self.tete(torch.cat([..., ...], dim=1))


bigru_pad = GRUSansEmpaquetage()
pad_justesse = entraine(..., "bigru_pad")
print(f"Séquences empaquetées : {bigru_justesse:.3f}, padding lu : {...:.3f}")
```

### Solution

In [ ]:
class GRUSansEmpaquetage(GRUTexte):
  """GRUTexte dont le GRU lit le padding : pas d'empaquetage."""

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    vecteurs = self.dropout(self.embedding(sequences))
    # Les 150 pas de chaque ligne, padding compris
    _, h_n = self.gru(vecteurs)
    return self.tete(torch.cat([h_n[-2], h_n[-1]], dim=1))


bigru_pad = GRUSansEmpaquetage()
pad_justesse = entraine(bigru_pad, "bigru_pad")
print(f"Séquences empaquetées : {bigru_justesse:.3f}, padding lu : {pad_justesse:.3f}")

Sans empaquetage, le GRU bidirectionnel fait moins bien : 53 % de justesse en test, contre 55 % avec les séquences empaquetées, et il apprend plus lentement (sa justesse de validation est plus basse à chaque epoch).

Son sens avant lit la phrase, puis ses 133 zéros : son dernier état caché est celui du dernier zéro, et ce qu'il a retenu de la phrase doit traverser plus de 130 pas sans information. Son sens arrière, lui, commence par les zéros, puis lit la phrase à l'envers : son dernier état est celui du premier mot, et il résume encore toute la phrase. Le *padding* gêne donc surtout le sens avant, et le modèle ne s'appuie plus guère que sur le sens arrière : sa justesse est à peu près celle d'un GRU d'un seul sens de lecture (partie 5). Chaque epoch dure aussi plus longtemps : le GRU calcule les 150 pas de chaque phrase, au lieu de ses seuls mots.

## 5. GRU simple ou bidirectionnel ?

Entraînez avec `entraine` un GRU simple de 64 unités, qui ne lit les phrases que dans un sens, et comparez-le au GRU bidirectionnel de la partie 3 :

- la justesse de test ;
- le nombre de paramètres de la couche récurrente, et du modèle entier ;
- la durée d'une epoch, que la barre de progression de Lightning affiche pendant l'entraînement.

Le second sens de lecture vaut-il son coût ici ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `GRUTexte(bidirectionnel=False)` : le même modèle, avec un GRU simple
- le résumé qu'affiche `entraine` donne le nombre de paramètres de chaque couche ; `sum(p.numel() for p in modele.parameters())` celui du modèle entier
- `bigru_justesse` contient la justesse de test du GRU bidirectionnel

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
gru = GRUTexte(...)
gru_justesse = entraine(..., "gru")
for modele, justesse in ((gru, ...), (bigru, bigru_justesse)):
  parametres = sum(p.numel() for p in modele.parameters())
  print(f"{modele.gru} : justesse de test {justesse:.3f}, {parametres} paramètres")
```

### Solution

In [ ]:
gru = GRUTexte(bidirectionnel=False)
gru_justesse = entraine(gru, "gru")
for modele, justesse in ((gru, gru_justesse), (bigru, bigru_justesse)):
  parametres = sum(p.numel() for p in modele.parameters())
  print(f"{modele.gru} : justesse de test {justesse:.3f}, {parametres} paramètres")

Le GRU simple fait un peu moins bien : 54 % de justesse en test, contre 55 % pour le GRU bidirectionnel. Ce point d'écart est de l'ordre de ce qui change d'une exécution à l'autre.

Le second sens de lecture double les paramètres de la couche récurrente (24 960 → 49 920), mais ce n'est que 2 % du modèle, dont l'embedding fait l'essentiel (1,28 million) ; une epoch dure à peine plus longtemps. Le gain est modeste, mais il ne coûte presque rien : l'état final d'un GRU simple résume surtout la fin de la phrase, et la direction arrière y ajoute un résumé de son début.

## 6. Des embeddings GloVe pré-entraînés et figés

Les embeddings des modèles précédents partent de vecteurs aléatoires et sont appris sur nos 41 000 phrases d'entraînement. Les vecteurs [GloVe](https://nlp.stanford.edu/projects/glove/), eux, ont été appris sur 6 milliards de mots (Wikipédia et Gigaword) : des mots de sens proche y ont des vecteurs proches.

Lisez le fichier `glove.6B.300d.txt` dans un dictionnaire `glove`, mot → vecteur : chaque ligne est un mot suivi de ses 300 coordonnées, séparés par des espaces, et seuls les mots de `vocabulaire` nous intéressent. La cellule qui suit la solution en fait une couche d'embedding.

Combien des 20 000 mots du vocabulaire GloVe connaît-il ? Lesquels manquent, et pourquoi ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `mot, *coordonnees = ligne.rstrip().split(" ")` sépare le mot de ses coordonnées ; `np.array(coordonnees, dtype="float32")` en fait un vecteur
- `set(vocabulaire)` : un test `mot in ...` rapide sur les 400 000 lignes du fichier

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
mots = set(vocabulaire)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as fichier:
  for ligne in fichier:
    mot, *coordonnees = ligne.rstrip().split(" ")
    if mot in ...:
      glove[mot] = np.array(coordonnees, dtype="float32")

manquants = [mot for mot in vocabulaire if mot not in ...]
print(f"{len(glove)} mots sur {len(vocabulaire)} dans GloVe")
print("Absents :", ", ".join(manquants[:30]))
```

### Solution

In [ ]:
# Seuls les vecteurs des mots du vocabulaire sont gardés
mots = set(vocabulaire)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as fichier:
  for ligne in fichier:
    mot, *coordonnees = ligne.rstrip().split(" ")
    if mot in mots:
      glove[mot] = np.array(coordonnees, dtype="float32")

manquants = [mot for mot in vocabulaire if mot not in glove]
print(f"{len(glove)} mots sur {len(vocabulaire)} dans GloVe")
print("Absents :", ", ".join(manquants[:30]))

# 19 114 des 20 000 mots du vocabulaire sont dans GloVe. Absents : le padding (<unk>, lui,
# a un vecteur dans GloVe), les types d'entités (ppperson, dddate…), des noms que spaCy
# n'a pas reconnus (holgrave, pyncheon), le vocabulaire de la chasse à la baleine
# (harpooneer, whalemen) et des graphies anciennes (d'ye, oftener) : des mots qu'un
# corpus du XXIe siècle (Wikipédia, la presse) n'emploie pas.

### La couche d'embedding GloVe

La cellule suivante construit `embedding_glove`, une couche `nn.Embedding` qui donne à chaque mot de `vocabulaire` son vecteur GloVe :

- `matrice_embedding`, de forme `(len(vocabulaire), 300)` : sa ligne `i` est le vecteur du mot `vocabulaire[i]`, ou reste nulle si GloVe ne connaît pas ce mot ;
- `nn.Embedding.from_pretrained` crée la couche à partir de cette matrice ; `freeze=True` la fige (ses vecteurs ne changent pas pendant l'entraînement), et `padding_idx=0` garde nul le vecteur du *padding*.

In [ ]:
matrice_embedding = np.zeros((len(vocabulaire), 300), dtype="float32")  # 0 si absent
for i, mot in enumerate(vocabulaire):
  if mot in glove:
    matrice_embedding[i] = glove[mot]
embedding_glove = nn.Embedding.from_pretrained(
  torch.from_numpy(matrice_embedding),
  freeze=True,  # table figée
  padding_idx=0,
)

### Entraîner avec GloVe

Entraînez le même GRU bidirectionnel que dans la partie 3, avec `embedding_glove` pour couche d'embedding, et comparez-le au premier :

- l'écart entre la justesse d'entraînement et celle de validation ;
- la justesse de test ;
- le nombre de paramètres entraînables, que le résumé du modèle donne.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`GRUTexte` prend la couche d'embedding par son premier paramètre, `embedding`. `bigru_justesse` contient la justesse de test du premier modèle.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bigru_glove = GRUTexte(...)
glove_justesse = entraine(..., "bigru_glove")
print(f"Embeddings appris : {bigru_justesse:.3f}, GloVe figé : {glove_justesse:.3f}")
```

#### Solution

In [ ]:
bigru_glove = GRUTexte(embedding_glove)
glove_justesse = entraine(bigru_glove, "bigru_glove")
print(f"Embeddings appris : {bigru_justesse:.3f}, GloVe figé : {glove_justesse:.3f}")

Avec GloVe figé, le modèle n'a plus que 141 705 paramètres entraînables, contre 1,3 million : les 6 millions de nombres de la table GloVe ne bougent pas.

Il généralise mieux : à la meilleure epoch, la 9e, l'écart entre la justesse d'entraînement et celle de validation n'est plus que de 9 points (70 % contre 61 %), au lieu de 15. Sa perte de validation descend plus bas, et la justesse de test passe de 55 % à 59 %. Les vecteurs GloVe apportent ce que nos 41 000 phrases ne peuvent pas apprendre seules : des mots de sens proche y ont des vecteurs proches, même quand ils sont rares dans le corpus.

L'écart finit pourtant par se creuser, à 73 % contre 60 % deux epochs plus tard : la couche récurrente, elle, apprend toujours par cœur. C'est le rôle du *dropout* du modèle suivant.

## 7. Un modèle plus profond

Un dernier modèle, donné : deux couches de GRU bidirectionnel empilées (`nb_couches=2`) sur les embeddings GloVe. La seconde lit la séquence des états cachés de la première, mot par mot.

`dropout=0.5` met à zéro, à chaque pas d'entraînement, la moitié des valeurs d'entrée de chaque couche récurrente (les embeddings, puis les états de la première couche), contre le surapprentissage.

In [ ]:
profond = GRUTexte(embedding_glove, nb_couches=2, dropout=0.5)
profond_justesse = entraine(profond, "bigru_profond")
print(
  f"GloVe et un GRU bidirectionnel : {glove_justesse:.3f}, deux : {profond_justesse:.3f}"
)

Le modèle profond atteint 60 % de justesse en test, à peine plus que le GRU bidirectionnel sur GloVe (59 %). Le *dropout* a supprimé le surapprentissage : à la meilleure epoch, la 13e, les justesses d'entraînement et de validation restent proches (60 % et 61 %), la première mesurée avec le *dropout* actif. Mais il ralentit l'apprentissage : la perte de validation baisse pendant 13 epochs, contre 9 sans *dropout*.

## 8. Les auteurs que le modèle confond

La cellule suivante affiche la matrice de confusion du modèle profond sur les phrases de test : la ligne d'un auteur donne la répartition de ses phrases entre les auteurs prédits.

In [ ]:
profond.eval()  # sans dropout
device = next(profond.parameters()).device
with torch.no_grad():
  y_pred = torch.cat(
    [profond(batch.to(device)).argmax(dim=1).cpu() for batch in X_test.split(1024)]
  )
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=encodeur_etiquettes.classes_,
  normalize="true",  # proportions de chaque ligne
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Auteur prédit")
ax.set_ylabel("Auteur réel")
plt.show()

### Lecture de la matrice

Quels auteurs le modèle reconnaît-il le mieux ? Lesquels confond-il, et qu'ont-ils en commun ? Les œuvres du corpus sont listées dans le [README du jeu de données](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Votre réponse ici

#### Solution

Dans notre exécution :

- Melville, Shaw et Wilde sont les mieux reconnus (65 % de leurs phrases), puis Cooper (61 %) : *Moby-Dick* a un vocabulaire à part (la mer, la chasse à la baleine), et les pièces de Shaw et de Wilde sont faites de répliques courtes, qui ne ressemblent pas à de la prose.
- Mais Shaw et Wilde se confondent entre eux (14 % et 18 % de leurs phrases) : deux auteurs irlandais de pièces de théâtre, de la même époque.
- Anne Brontë est prise pour sa sœur Charlotte dans 20 % des cas : deux romancières anglaises de la même époque, aux romans racontés à la première personne, par une gouvernante dans *Agnes Grey* comme dans *Jane Eyre*. Charlotte, presque deux fois plus représentée, attire les phrases ambiguës de tous les auteurs : sa colonne est la plus chargée après la diagonale.
- Emerson et Thoreau aussi (18 % et 11 %) : deux essayistes américains du transcendantalisme, voisins et amis.

Une phrase isolée est un indice faible : beaucoup sont courtes et banales (« Yes . »), et aucun modèle ne peut les attribuer. Le modèle profond (60 % en test) ne fait guère mieux que le GRU bidirectionnel sur GloVe (59 %), et une régression logistique sur une représentation TF-IDF des phrases atteint environ 61 % sur ce même découpage : sur des phrases courtes, les mots employés comptent plus que leur ordre.